# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to download raw MS MRI datasets and prepare the environment for 3D Deep Learning.

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive for saving model weights.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel tqdm

from google.colab import drive
import os

drive.mount('/content/drive')

# Define Workspace Paths
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
MODEL_SAVE_PATH = "/content/drive/MyDrive/MS_Project/models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"✅ Workspace initialized. Models will be saved to: {MODEL_SAVE_PATH}")

## 📊 1.2 Data Ingestion (Hybrid Strategy)
We pull the renamed ZIP files from your **Google Drive** and extract them to **local Colab storage** (`/content/data`) for maximum training speed.

In [ ]:
import os
import glob
from google.colab import drive

# 1. Setup Paths
drive.mount('/content/drive')
DRIVE_DATA_ROOT = "/content/drive/MyDrive/brain_dataset"
LOCAL_DATA_DIR = "/content/data/raw"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# 2. Categories to Extract
categories = ["MSLesSeg", "Mendeley", "Long-MR-MS"]

for category in categories:
    drive_folder = os.path.join(DRIVE_DATA_ROOT, category)
    local_folder = os.path.join(LOCAL_DATA_DIR, category)
    os.makedirs(local_folder, exist_ok=True)
    
    print(f"\n📦 Processing {category}...")
    
    # Find every ZIP file in the folder (handles independent parts like MSLesSeg_Part1..4)
    zips = glob.glob(os.path.join(drive_folder, "*.zip"))
    
    if not zips:
        print(f"  ⚠️ No ZIP files found in {drive_folder}")
        continue
        
    for zip_path in zips:
        print(f"  ⚡ Extracting {os.path.basename(zip_path)} to local disk...")
        # unzip each file individually as they are independent
        !unzip -q -o "{zip_path}" -d "{local_folder}"

print("\n🚀 SUCCESS: Fast local data storage is ready for training.")

## 🔍 1.2.1 Directory Mapping (Diagnostic)
Run this cell to see exactly where your FLAIR and Mask files are located inside the subfolders.

In [ ]:
import os

def map_directory_structure(root_dir, max_depth=3):
    print(f"🔍 Mapping structure for: {root_dir}\n")
    for root, dirs, files in os.walk(root_dir):
        depth = root.replace(root_dir, '').count(os.sep)
        if depth <= max_depth:
            indent = '  ' * depth
            print(f"{indent}📂 {os.path.basename(root)}/")
            if files:
                for f in files[:3]:
                    print(f"{indent}  📄 {f}")
                if len(files) > 3:
                    print(f"{indent}  ... ({len(files)-3} more files)")

# Run the mapping
map_directory_structure("/content/data/raw")

## 📂 1.2.2 Data Discovery & Pairing
This cell pairs MRI scans with their Ground Truth masks for training across all datasets.

In [ ]:
import glob

def create_data_list(raw_dir):
    data_list = []
    
    # 1. Handle Long-MR-MS (Specific Patient Mapping)
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        print("🔎 Searching Long-MR-MS...")
        patient_folders = glob.glob(os.path.join(long_path, 'patient*'))
        for p_folder in patient_folders:
            # Images usually end in FLAIRreg.nii.gz, Labels end in _gt.nii.gz
            gt_files = glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flair_files = glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            
            if gt_files and flair_files:
                for f in flair_files:
                    data_list.append({'image': f, 'label': gt_files[0]})
    
    # 2. Handle Mendeley & MSLesSeg (Flexible Recursive Mapping)
    # We look for files containing 'FLAIR' and match with 'mask' or 'GT'
    print("🔎 Searching for other datasets (MSLesSeg/Mendeley)...")
    # Note: These regex patterns will be refined once you run 1.2.1 and see the names
    
    print(f"✅ Total Image-Label pairs discovered: {len(data_list)}")
    return data_list

train_files = create_data_list("/content/data/raw")

## 🧠 1.3 The Preprocessing Engine
Standardizing MRI volumes to 1mm isotropic spacing and RAS orientation.

In [ ]:
import SimpleITK as sitk
import numpy as np

def preprocess_volume(img_path, target_spacing=(1.0, 1.0, 1.0), is_mask=False):
    img = sitk.ReadImage(img_path)
    img = sitk.DICOMOrient(img, "RAS")
    
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    new_size = [int(round(osz * ospc / tspc)) for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)]
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline)
    img = resampler.Execute(img)
    
    if not is_mask:
        mask = sitk.OtsuThreshold(img, 0, 1)
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        img = corrector.Execute(img, mask)
        array = sitk.GetArrayFromImage(img)
        mean, std = np.mean(array), np.std(array)
        array = (array - mean) / (std + 1e-8)
        img_out = sitk.GetImageFromArray(array)
        img_out.CopyInformation(img)
        return img_out
    return img

print("✅ Preprocessing logic implemented.")

## 🧪 1.4 Architecture: 3D Residual U-Net
We implement a 3D Residual U-Net using MONAI, optimized for small batch sizes with InstanceNorm.

In [ ]:
from monai.networks.nets import UNet
import torch

def get_model(in_channels=1, out_channels=1):
    model = UNet(
        spatial_dims=3,
        in_channels=in_channels,
        out_channels=out_channels,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="INSTANCE",
        act="LEAKYRELU",
        dropout=0.1
    )
    return model

print("✅ 3D U-Net Architecture loaded.")

## 📉 1.5 Loss & Metrics
Hybrid Dice + Weighted BCE for small-lesion sensitivity.

In [ ]:
from monai.losses import DiceLoss
import torch.nn as nn
from scipy.ndimage import label

class MSLesionLoss(nn.Module):
    def __init__(self, alpha=1.0, weight=10.0):
        super(MSLesionLoss, self).__init__()
        self.alpha = alpha
        self.dice_loss = DiceLoss(sigmoid=True, squared_pred=True)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]).cuda())

    def forward(self, input, target):
        return self.dice_loss(input, target) + self.alpha * self.bce_loss(input, target)

def get_lesion_wise_metrics(y_pred, y_true):
    pred_labels, n_pred = label(y_pred)
    true_labels, n_true = label(y_true)
    if n_true == 0: return {'recall': 0, 'precision': 0, 'fp': n_pred}
    tp = sum([1 for i in range(1, n_true+1) if np.any(y_pred[true_labels == i])])
    recall = tp / n_true
    fp = sum([1 for i in range(1, n_pred+1) if not np.any(y_true[pred_labels == i])])
    precision = (n_pred - fp) / n_pred if n_pred > 0 else 0
    return {'recall': recall, 'precision': precision, 'fp': fp}

print("✅ Loss and Metrics logic implemented.")

## 🚀 1.6 The Full Training Engine
Executes the actual 3D training loop with Balanced Sampling and AMP.

In [ ]:
from monai.data import CacheDataset, DataLoader
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, RandCropByPosNegLabeld, ToTensord)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

def train_model(data_list, epochs=50):
    device = torch.device("cuda")
    model = get_model().to(device)
    loss_fn = MSLesionLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()
    
    # MONAI Transforms with Balanced Sampling (50% Lesion patches)
    transforms = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        RandCropByPosNegLabeld(
            keys=["image", "label"], 
            label_key="label", 
            spatial_size=(96,96,96), 
            pos=1, neg=1, num_samples=4
        ),
        ToTensord(keys=["image", "label"])
    ])
    
    # Using CacheDataset for maximum Colab performance
    ds = CacheDataset(data=data_list, transform=transforms, cache_rate=1.0)
    loader = DataLoader(ds, batch_size=1, shuffle=True)
    
    print("🚀 Starting Training Engine on T4 GPU...")
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = batch["image"].to(device), batch["label"].to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
        
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        print(f"🔥 Epoch {epoch+1} Complete. Average Loss: {avg_loss:.4f}")
        
        # Persistence: Save to Drive every 5 epochs
        if (epoch + 1) % 5 == 0:
            save_path = os.path.join(MODEL_SAVE_PATH, f"model_epoch_{epoch+1}.pth")
            torch.save(model.state_dict(), save_path)
            # Also save latest as 'best_model.pth'
            torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, "best_model.pth"))
            print(f"💾 Checkpoint saved to: {save_path}")

print("✅ Training Engine Ready.")

# To launch: 
# train_model(train_files)